# Timefolio Engine — Quickstart

This notebook walks through the most common use-cases of the `timefolio` library.

**Prerequisites**

```bash
# Install the library from the project root
pip install -e .            # core (requests only)
pip install -e .[quant]    # core + data/quant extras
```

**Credentials**

Create a `.env` file in the project root (never commit this file):

```
TIMEFOLIO_EMAIL=you@example.com
TIMEFOLIO_PASSWORD=your_password
TIMEFOLIO_PF_ID=18762
```

Or set the environment variables directly before running this notebook.

## 0. Setup

In [ ]:
import logging
import os

# Optional: load credentials from a .env file in the project root.
# Remove this block if you prefer to set env vars another way.
try:
    from dotenv import load_dotenv
    load_dotenv()  # reads ../.env relative to examples/
    load_dotenv(dotenv_path="../.env", override=False)
except ImportError:
    pass  # python-dotenv is optional; credentials can come from the shell env

# Show INFO-level log messages from the library so you can follow what's happening.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

from timefolio import TimefolioAPIClient, TimefolioTrader

## 1. Authentication

`TimefolioAPIClient` manages a single `requests.Session`.  
Call `login()` once — the Bearer token is automatically attached to every
subsequent request.

In [ ]:
EMAIL    = os.environ["TIMEFOLIO_EMAIL"]
PASSWORD = os.environ["TIMEFOLIO_PASSWORD"]
PF_ID    = int(os.environ.get("TIMEFOLIO_PF_ID", 18762))

api = TimefolioAPIClient(email=EMAIL, password=PASSWORD)

if not api.login():
    raise RuntimeError("Login failed — check your credentials in .env")

print(f"Logged in.  Active portfolio ID: {PF_ID}")

## 2. Creating a Trader

`TimefolioTrader` wraps the authenticated client with trading-specific methods.

In [ ]:
trader = TimefolioTrader(api_client=api, pf_id=PF_ID)
print(f"Trader ready (pfId={trader.pf_id})")

## 3. Switch Tournament (optional)

If you are enrolled in multiple contests, use `set_tournament()` to switch
the active portfolio by matching a substring of the contest name.

In [ ]:
# Example — change "연습용 대회" to the actual contest name you want to trade in.
# trader.set_tournament(target_name="연습용 대회")
# print(f"Now using pfId={trader.pf_id}")

## 4. Portfolio Balance

Fetch the current portfolio summary: holdings, cash position, NAV, and P&L.

In [ ]:
balance = trader.get_balance()
balance  # raw JSON — structure depends on the server response

## 5. Market Order (Immediate Execution)

Omit `hm0` / `hm1` to execute at the current market price right now.

| Parameter | Description |
|---|---|
| `prod_id` | Ticker with `A` prefix (e.g. `A005930` = Samsung Electronics) |
| `weight` | Fraction of total NAV — `0.05` = 5 % |
| `ls` | `"L"` Buy / `"S"` Sell |
| `limit_idx` | Order-book aggressiveness 1–10 (default 5) |

In [ ]:
# Buy Samsung Electronics (A005930) at 5 % portfolio weight, execute immediately.
result = trader.order(
    prod_id="A005930",
    weight=0.05,
    ls="L",
)
print(result)

## 6. Scheduled / TWAP Order

Set `hm0` (start time) and `hm1` (end time) to enable **TWAP** (Time-Weighted
Average Price) execution.  The server slices the order across the given window.

You can also supply `target_date` to schedule on a future business day.

In [ ]:
# Buy Hyundai Motor (A005380) at 10 % weight.
# Execution is spread from 09:00 to 12:20 on the next trading day.
result = trader.order(
    prod_id="A005380",
    weight=0.10,
    ls="L",
    hm0="09:00",
    hm1="12:20",
    target_date="2026-03-03",  # must be a business day
)
print(result)

## 7. Limit Order

Pass `limit_prc` to peg the order to a specific price.  
The server will not execute above (buy) or below (sell) this price.

In [ ]:
# Buy Samsung Electronics at a limit price of 55,000 KRW.
result = trader.order(
    prod_id="A005930",
    weight=0.05,
    ls="L",
    limit_prc=55_000,
)
print(result)

## 8. Stop Order

Pass `stop_prc` to set a stop-loss or stop-breakout trigger.  
The order activates only when the market price crosses `stop_prc`.

In [ ]:
# Sell Samsung Electronics if the price drops to 50,000 KRW (stop-loss).
result = trader.order(
    prod_id="A005930",
    weight=0.05,
    ls="S",
    stop_prc=50_000,
)
print(result)

## 9. Limit + Stop (Bracket Order)

Combine both parameters for a bracket: the order only activates after
`stop_prc` is touched and is then capped by `limit_prc`.

In [ ]:
# Enter a long position in KOSPI ETF (A069500) only if price breaks above
# 32,000, and cap the buy price at 32,500.
result = trader.order(
    prod_id="A069500",
    weight=0.08,
    ls="L",
    limit_prc=32_500,
    stop_prc=32_000,
    hm0="09:00",
    hm1="11:30",
)
print(result)

## 10. Cancel an Order

Use the `ordId` returned in any successful `order()` response to cancel it.

In [ ]:
# Replace 99999 with a real ordId from a previous order response.
# cancel_result = trader.cancel_order(ord_id=99999)
# print(cancel_result)

## 11. Raw API Access

`TimefolioAPIClient` exposes `get()` and `post()` helpers for any endpoint
not yet wrapped by `TimefolioTrader`.

In [ ]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

# Example: raw GET to fetch portfolio details
res = api.get("Portfolio/Summary", params={"pfId": PF_ID, "d": today})
print(res.status_code)
# res.json()  # full response payload

---

## Parameter Reference

| Parameter | Type | Default | Description |
|---|---|---|---|
| `prod_id` | `str` | — | Ticker symbol, e.g. `"A005930"` |
| `weight` | `float` | — | Portfolio weight, `0.05` = 5 % |
| `ls` | `str` | `"L"` | `"L"` Long/Buy · `"S"` Short/Sell |
| `ex` | `str` | `"E"` | Execution algorithm type |
| `limit_idx` | `int` | `5` | Order-book depth aggressiveness (1–10) |
| `limit_prc` | `float\|None` | `None` | Exact limit price; `None` = algo/market |
| `stop_prc` | `float\|None` | `None` | Stop trigger price; `None` = disabled |
| `hm0` | `str\|None` | `None` | Start time `"HH:MM"`; `None` = immediate |
| `hm1` | `str\|None` | `None` | End time `"HH:MM"` for TWAP window |
| `target_date` | `str\|None` | `None` | Business date `"YYYY-MM-DD"`; `None` = today |